In [1]:
import numpy as np
import pandas as pd

In [2]:
from kloppy import skillcorner

match_id = 1899585

# dataset = skillcorner.load(
#     meta_data=f"https://raw.githubusercontent.com/SkillCorner/opendata/741bdb798b0c1835057e3fa77244c1571a00e4aa/data/matches/{match_id}/{match_id}_match.json",
#     raw_data=f"https://media.githubusercontent.com/media/SkillCorner/opendata/741bdb798b0c1835057e3fa77244c1571a00e4aa/data/matches/{match_id}/{match_id}_tracking_extrapolated.jsonl",
#     # Optional arguments
#     sample_rate=1 / 10,
#     limit=100,
#     coordinates="skillcorner",
#     include_empty_frames=False,
# )
# pd.set_option('display.max_columns', None)
# df = dataset.to_df().copy()

In [3]:
from data_ingestor_github import SkillCornerDataIngestor
di = SkillCornerDataIngestor()
meta_data = di._get_bronze_meta_data(match_id=match_id)
bronze_tracking_data = di._get_bronze_tracking_data(match_id=match_id)
tracking_data = di._get_silver_tracking_data(bronze_tracking_data)

In [4]:
player_data = []

for player in meta_data["players"]:
    player_data.append({
        "player_id": player["id"],
        "team_id": player["team_id"],
        "position": player["player_role"]["acronym"],
        "name": player["short_name"],
        "databallpy_id": (
            f"home_{player['id']}"
            if player["team_id"] == 867
            else f"away_{player['id']}"
        )
    })

In [5]:
def get_vx(tracking_data, frame_num, player_id, dt=0.1):
    current = tracking_data[
        (tracking_data["frame"] == frame_num) &
        (tracking_data["player_id"] == player_id)
    ][["x"]]

    previous = tracking_data[
        (tracking_data["frame"] == frame_num - 1) &
        (tracking_data["player_id"] == player_id)
    ][["x"]]

    if current.empty or previous.empty:
        return np.nan

    dx = current.iloc[0]["x"] - previous.iloc[0]["x"]
    return dx / dt


def get_vy(tracking_data, frame_num, player_id, dt=0.1):
    current = tracking_data[
        (tracking_data["frame"] == frame_num) &
        (tracking_data["player_id"] == player_id)
    ][["y"]]

    previous = tracking_data[
        (tracking_data["frame"] == frame_num - 1) &
        (tracking_data["player_id"] == player_id)
    ][["y"]]

    if current.empty or previous.empty:
        return np.nan

    dy = current.iloc[0]["y"] - previous.iloc[0]["y"]
    return dy / dt

In [7]:
player_data

[{'player_id': 957734,
  'team_id': 867,
  'position': 'LWB',
  'name': 'C. Piper',
  'databallpy_id': 'home_957734'},
 {'player_id': 27003,
  'team_id': 867,
  'position': 'CF',
  'name': 'K. Nagasawa',
  'databallpy_id': 'home_27003'},
 {'player_id': 14736,
  'team_id': 4177,
  'position': 'RM',
  'name': 'L. Verstraete',
  'databallpy_id': 'away_14736'},
 {'player_id': 6799,
  'team_id': 867,
  'position': 'CF',
  'name': 'M. Rojas',
  'databallpy_id': 'home_6799'},
 {'player_id': 965684,
  'team_id': 4177,
  'position': 'LW',
  'name': 'L. Toomey',
  'databallpy_id': 'away_965684'},
 {'player_id': 26969,
  'team_id': 867,
  'position': 'LF',
  'name': 'H. Ishige',
  'databallpy_id': 'home_26969'},
 {'player_id': 43829,
  'team_id': 4177,
  'position': 'RW',
  'name': 'N. Moreno',
  'databallpy_id': 'away_43829'},
 {'player_id': 799092,
  'team_id': 867,
  'position': 'RCB',
  'name': 'M. Sheridan',
  'databallpy_id': 'home_799092'},
 {'player_id': 133501,
  'team_id': 4177,
  'posi

In [8]:
frame_num = 20
new_row = {"frame": frame_num}

current_frame = tracking_data[tracking_data["frame"] == frame_num].set_index("player_id")

for player in player_data:
    player_id = player["player_id"]
    player_databallpy_id = player["databallpy_id"]

    if player_id not in current_frame.index:
        continue

    new_row[f"{player_databallpy_id}_x"] = current_frame.loc[player_id, "x"]
    new_row[f"{player_databallpy_id}_y"] = current_frame.loc[player_id, "y"]
    new_row[f"{player_databallpy_id}_vx"] = get_vx(tracking_data, frame_num, player_id)
    new_row[f"{player_databallpy_id}_vy"] = get_vy(tracking_data, frame_num, player_id)

# ball_x and ball_y are columns on each player row, so just take one row from the frame
frame_rows = tracking_data[tracking_data["frame"] == frame_num]
new_row["ball_x"] = frame_rows["ball_x"].iloc[0]
new_row["ball_y"] = frame_rows["ball_y"].iloc[0]

frame = pd.DataFrame([new_row])
frame.head()

,frame,home_27003_x,home_27003_y,home_27003_vx,home_27003_vy,away_14736_x,away_14736_y,away_14736_vx,away_14736_vy,home_26969_x,...,away_51667_x,away_51667_y,away_51667_vx,away_51667_vy,away_285188_x,away_285188_y,away_285188_vx,away_285188_vy,ball_x,ball_y
0,20,-6.47,6.21,1.9,0.0,8.45,2.9,-1.6,-0.9,1.62,...,21.13,5.66,-0.8,-0.8,42.55,0.2,0.4,-0.1,-11.79,-0.11


In [9]:
from databallpy.features import(
    get_approximate_voronoi,
    get_pitch_control,
    get_pitch_control_single_frame,
    get_team_influence,
    get_player_influence,
)

In [10]:
pitch_dimensions = (106, 68)

pitch_control = get_pitch_control_single_frame(frame.iloc[0], pitch_dimensions, pitch_dimensions[0], pitch_dimensions[1])

In [11]:
pitch_control

array([[0.49970425, 0.49966765, 0.49963027, ..., 0.50099154, 0.50081596,
        0.50067398],
       [0.49958156, 0.49952981, 0.49947697, ..., 0.50129523, 0.50107608,
        0.50089698],
       [0.49941403, 0.49934161, 0.49926767, ..., 0.50168641, 0.50141404,
        0.50118891],
       ...,
       [0.49909547, 0.49898419, 0.49887073, ..., 0.50203409, 0.50169621,
        0.50141642],
       [0.49934512, 0.49926454, 0.49918235, ..., 0.50157822, 0.50130483,
        0.50108061],
       [0.49953072, 0.49947296, 0.49941403, ..., 0.50122021, 0.5010001 ,
        0.50082118]], shape=(68, 106))

In [13]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# If inline notebook rendering still fails, use:
# pio.renderers.default = "browser"

pitch_length = 106
pitch_width = 68

row = frame.iloc[0]

x = np.linspace(-pitch_length / 2, pitch_length / 2, pitch_control.shape[1])
y = np.linspace(-pitch_width / 2, pitch_width / 2, pitch_control.shape[0])

home_x = []
home_y = []
away_x = []
away_y = []

for col in row.index:
    if col.startswith("home_") and col.endswith("_x"):
        player_base = col[:-2]
        y_col = f"{player_base}_y"
        if y_col in row.index:
            home_x.append(row[col])
            home_y.append(row[y_col])

    elif col.startswith("away_") and col.endswith("_x"):
        player_base = col[:-2]
        y_col = f"{player_base}_y"
        if y_col in row.index:
            away_x.append(row[col])
            away_y.append(row[y_col])

ball_x = row["ball_x"]
ball_y = row["ball_y"]

fig = go.Figure()

fig.add_trace(
    go.Contour(
        z=pitch_control,
        x=x,
        y=y,
        colorscale="RdBu",
        contours=dict(
            coloring="heatmap",
            showlabels=True,
            labelfont=dict(size=10, color="black"),
        ),
        line=dict(width=1),
        colorbar=dict(title="Pitch Control"),
        name="Pitch Control",
    )
)

fig.add_trace(
    go.Scatter(
        x=home_x,
        y=home_y,
        mode="markers",
        marker=dict(size=10, color="blue", line=dict(color="white", width=1)),
        name="Home Players",
    )
)

fig.add_trace(
    go.Scatter(
        x=away_x,
        y=away_y,
        mode="markers",
        marker=dict(size=10, color="red", line=dict(color="white", width=1)),
        name="Away Players",
    )
)

fig.add_trace(
    go.Scatter(
        x=[ball_x],
        y=[ball_y],
        mode="markers",
        marker=dict(size=9, color="black", symbol="circle"),
        name="Ball",
    )
)

# Pitch outline
fig.add_shape(
    type="rect",
    x0=-pitch_length / 2,
    x1=pitch_length / 2,
    y0=-pitch_width / 2,
    y1=pitch_width / 2,
    line=dict(color="black", width=2),
)

# Halfway line
fig.add_shape(
    type="line",
    x0=0,
    x1=0,
    y0=-pitch_width / 2,
    y1=pitch_width / 2,
    line=dict(color="black", width=1),
)

fig.update_layout(
    title="Pitch Control with Players and Ball",
    xaxis=dict(
        title="Pitch X",
        range=[-pitch_length / 2, pitch_length / 2],
        scaleanchor="y",
    ),
    yaxis=dict(
        title="Pitch Y",
        range=[-pitch_width / 2, pitch_width / 2],
    ),
    width=900,
    height=600,
    template="plotly_white",
)

fig.show()